# Phân tích khám phá dữ liệu (EDA) cho hai nguồn cảnh báo nguy cơ bỏ học

Notebook này chỉ sử dụng hai file dữ liệu đã được nhóm thống nhất:

- `student_dropout_and_success.csv`: dữ liệu học vụ, tài chính và kết quả học kỳ.
- `student_dropout.csv`: dữ liệu GPA, chuyên cần, stress và hành vi học tập.

Hai nguồn được phân tích độc lập và **không ghép theo từng dòng hoặc theo sinh viên**.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# Hỗ trợ chạy notebook từ thư mục notebooks hoặc từ thư mục gốc của kho mã.
data_candidates = [Path("../data"), Path("data")]
DATA_DIR = next((path for path in data_candidates if path.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError("Không tìm thấy thư mục data. Hãy chạy notebook từ thư mục notebooks hoặc thư mục gốc của kho mã.")

# Tạo thư mục lưu các báo cáo phân tích nếu thư mục này chưa tồn tại.
OUTPUT_DIR = DATA_DIR.parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Khai báo đường dẫn của hai nguồn dữ liệu đã được nhóm thống nhất.
SOURCE_1_PATH = DATA_DIR / "student_dropout_and_success.csv"
SOURCE_2_PATH = DATA_DIR / "student_dropout.csv"

print("Thư mục dữ liệu:", DATA_DIR.resolve())
print("Thư mục kết quả:", OUTPUT_DIR.resolve())

## 1. Đọc hai nguồn dữ liệu

In [ ]:
# Đọc riêng từng nguồn; không ghép hai bảng vì không có khóa sinh viên chung.
source_1 = pd.read_csv(SOURCE_1_PATH)
source_2 = pd.read_csv(SOURCE_2_PATH)

print("student_dropout_and_success.csv:", source_1.shape)
print("student_dropout.csv:", source_2.shape)

# Dừng sớm nếu file dữ liệu bị thay đổi hoặc đọc không đúng cấu trúc dự kiến.
assert source_1.shape == (4424, 35), f"Kích thước Nguồn 1 không đúng: {source_1.shape}"
assert source_2.shape == (10000, 19), f"Kích thước Nguồn 2 không đúng: {source_2.shape}"

In [ ]:
print("\nCác cột của student_dropout_and_success.csv:")
print(source_1.columns.tolist())

print("\nCác cột của student_dropout.csv:")
print(source_2.columns.tolist())

print("\nHai dòng đầu của Nguồn 1:")
print(source_1.head(2).to_string(index=False))

print("\nHai dòng đầu của Nguồn 2:")
print(source_2.head(2).to_string(index=False))

## 2. Tổng quan và giá trị thiếu

In [ ]:
# Gom hai DataFrame vào một từ điển để áp dụng cùng một quy trình kiểm tra.
sources = {
    "student_dropout_and_success.csv": source_1,
    "student_dropout.csv": source_2,
}

summary_rows = []
missing_rows = []

# Thống kê kích thước, số dòng trùng và tổng số giá trị thiếu của từng nguồn.
for data_source, frame in sources.items():
    summary_rows.append({
        "data_source": data_source,
        "rows": len(frame),
        "columns": frame.shape[1],
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_values": int(frame.isna().sum().sum()),
    })

    # Chỉ đưa vào báo cáo những cột thực sự có giá trị thiếu.
    for column, missing_count in frame.isna().sum().items():
        if missing_count > 0:
            missing_rows.append({
                "data_source": data_source,
                "column": column,
                "missing_count": int(missing_count),
                "missing_percent": round(missing_count / len(frame) * 100, 2),
            })

eda_summary = pd.DataFrame(summary_rows)
missing_report = pd.DataFrame(
    missing_rows,
    columns=["data_source", "column", "missing_count", "missing_percent"],
)

print(eda_summary.to_string(index=False))
print("\nCác cột có giá trị thiếu:")
print(missing_report.to_string(index=False) if not missing_report.empty else "Không có giá trị thiếu")

## 3. Phân phối biến mục tiêu

In [ ]:
# Tính số lượng và tỷ lệ phần trăm của từng nhãn trong biến mục tiêu.
def target_distribution(frame, target_column, data_source):
    counts = frame[target_column].value_counts(dropna=False)
    percentages = frame[target_column].value_counts(dropna=False, normalize=True).mul(100)

    return pd.DataFrame({
        "data_source": data_source,
        "target_value": counts.index.astype(str),
        "count": counts.values,
        "percent": percentages.reindex(counts.index).round(2).values,
    })

target_source_1 = target_distribution(
    source_1, "Target", "student_dropout_and_success.csv"
)
target_source_2 = target_distribution(
    source_2, "Dropout", "student_dropout.csv"
)
target_report = pd.concat([target_source_1, target_source_2], ignore_index=True)

print(target_report.to_string(index=False))

## 4. Tạo biến mục tiêu nhị phân dự kiến cho Nguồn 1

Trong thí nghiệm nhị phân chính, giữ `Dropout` và `Graduate`; tạm loại `Enrolled` vì đây là trạng thái chưa có kết quả cuối cùng. File dữ liệu gốc không bị thay đổi.

In [ ]:
# Tạo bản sao phục vụ thí nghiệm nhị phân; dữ liệu gốc vẫn được giữ nguyên.
source_1_binary = source_1[
    source_1["Target"].isin(["Dropout", "Graduate"])
].copy()

# Quy ước: sinh viên bỏ học = 1, sinh viên tốt nghiệp = 0.
source_1_binary["Dropout"] = (
    source_1_binary["Target"] == "Dropout"
).astype(int)

excluded_enrolled = int((source_1["Target"] == "Enrolled").sum())

print("Số dòng dùng cho thí nghiệm nhị phân:", len(source_1_binary))
print("Số dòng Enrolled tạm loại:", excluded_enrolled)
print(source_1_binary["Dropout"].value_counts().sort_index().to_string())

assert len(source_1_binary) + excluded_enrolled == len(source_1)

## 5. Kiểm tra miền giá trị và giá trị phân loại

In [ ]:
# Khai báo miền giá trị hợp lý cho các cột số quan trọng của từng nguồn.
range_rules = {
    "student_dropout_and_success.csv": {
        "Age at enrollment": (0, None),
        "Curricular units 1st sem (grade)": (0, 20),
        "Curricular units 2nd sem (grade)": (0, 20),
        "Curricular units 1st sem (enrolled)": (0, None),
        "Curricular units 1st sem (approved)": (0, None),
        "Curricular units 2nd sem (enrolled)": (0, None),
        "Curricular units 2nd sem (approved)": (0, None),
    },
    "student_dropout.csv": {
        "Age": (0, None),
        "GPA": (0, 4),
        "Semester_GPA": (0, 4),
        "CGPA": (0, 4),
        "Attendance_Rate": (0, 100),
        "Stress_Index": (0, 10),
        "Study_Hours_per_Day": (0, 24),
        "Assignment_Delay_Days": (0, None),
    },
}

# Đếm các giá trị nằm ngoài miền hợp lý; bỏ qua giá trị thiếu ở bước này.
quality_rows = []
for data_source, rules in range_rules.items():
    frame = sources[data_source]
    for column, (minimum, maximum) in rules.items():
        series = frame[column].dropna()
        invalid = series < minimum
        if maximum is not None:
            invalid = invalid | (series > maximum)
        quality_rows.append({
            "data_source": data_source,
            "column": column,
            "observed_min": float(series.min()),
            "observed_max": float(series.max()),
            "invalid_count": int(invalid.sum()),
        })

quality_report = pd.DataFrame(quality_rows)
print(quality_report.to_string(index=False))

In [ ]:
# Xem các giá trị xuất hiện trong những cột phân loại của Nguồn 2.
categorical_columns_source_2 = [
    "Gender",
    "Internet_Access",
    "Part_Time_Job",
    "Scholarship",
    "Semester",
    "Department",
    "Parental_Education",
]

for column in categorical_columns_source_2:
    values = source_2[column].value_counts(dropna=False)
    print(f"\n{column}:")
    print(values.to_string())

## 6. Vai trò của hai nguồn

- `student_dropout_and_success.csv` cung cấp góc nhìn học vụ, tài chính và kết quả học kỳ.
- `student_dropout.csv` cung cấp góc nhìn GPA, chuyên cần, stress và hành vi học tập.
- Hai nguồn không có khóa sinh viên chung nên không được ghép theo từng dòng.
- Mỗi nguồn sẽ được chia dữ liệu và đánh giá độc lập bằng Học máy (Machine Learning) và Chấm điểm dựa trên luật (Rule-based Scoring).
- Chỉ tổng hợp kết quả tại tầng so sánh và hỗ trợ ra quyết định.

## 7. Xuất báo cáo phân tích khám phá dữ liệu

In [ ]:
# Lưu các kết quả trung gian để nhóm dùng lại trong báo cáo và các bước sau.
eda_summary.to_csv(
    OUTPUT_DIR / "eda_sources_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
missing_report.to_csv(
    OUTPUT_DIR / "eda_missing_values.csv",
    index=False,
    encoding="utf-8-sig",
)
target_report.to_csv(
    OUTPUT_DIR / "eda_target_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)
quality_report.to_csv(
    OUTPUT_DIR / "eda_data_quality.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Đã xuất:")
for filename in [
    "eda_sources_summary.csv",
    "eda_missing_values.csv",
    "eda_target_distribution.csv",
    "eda_data_quality.csv",
]:
    print("-", OUTPUT_DIR / filename)

## 8. Kết luận Bước 2

Notebook đã xác nhận cấu trúc, giá trị thiếu, phân phối biến mục tiêu và chất lượng cơ bản của hai nguồn. Các bước chọn ngưỡng chấm điểm dựa trên luật, xử lý giá trị thiếu chính thức và huấn luyện mô hình chỉ được thực hiện sau khi tạo cố định tập huấn luyện, tập xác thực và tập kiểm thử.